# Module 5 — Inverse Kinematics
## Unit 2 — Inverse Kinematics of One and Two Joints
### Lesson 2.2 — The Planar Two-Link Arm: Geometry of the Solution

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Law of cosines gives the elbow; reach sets the bend.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    """Forward kinematics: planar 2-link gripper position."""
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def reachable(x, y, L1, L2, tol=1e-9):
    r = np.hypot(x, y)
    return abs(L1-L2)-tol <= r <= L1+L2+tol

def ik_two_link(x, y, L1, L2):
    """Return both (theta1,theta2) solutions; [] if unreachable, one if on boundary."""
    r2 = x*x + y*y
    c2 = (r2 - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        t2 = sign*np.arccos(c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1+L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(np.sin(t2)) < 1e-6:    # boundary: the two coincide
            break
    return sols

L1,L2=0.4,0.3
def elbow_cosine(x,y,L1,L2):
    return (x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
for tgt in [(0.7,0),(0.5,0),(0.1,0)]:
    c=elbow_cosine(*tgt,L1,L2)
    note="straight" if abs(c-1)<1e-9 else ("folded" if abs(c+1)<1e-9 else "bent")
    print(tgt,"cos(theta2)=",round(c,3),note)


## Coding Exercise (§8)
Compute cos(theta2), flag out-of-range (unreachable).


In [ ]:
# YOUR CODE HERE
c=elbow_cosine(0.5,0,L1,L2); assert abs(c-0.0)<1e-9     # right-angle elbow
assert elbow_cosine(0.7,0,L1,L2) > 1-1e-9               # straight (boundary)
print("law-of-cosines OK")


## Check your work


In [ ]:
# the worked-example elbow-down solution
sols=ik_two_link(0.5,0.0,L1,L2)
t1,t2=sols[0]
assert np.allclose(fk_two_link(t1,t2,L1,L2),[0.5,0.0],atol=1e-9)
assert abs(np.degrees(t2)-90)<1e-6 or abs(np.degrees(t2)+90)<1e-6
print("All checks passed.")
